# Part 3 - CatBoost 공실률 예측 모델

`data/processed/ktx_model_input.csv`를 사용해 CatBoost 회귀 모델을 학습한다. 시계열 데이터이므로 일반 KFold는 사용하지 않고 `TimeSeriesSplit` 기반 Walk-forward Cross Validation으로 평가한다.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit

from src.model import (
    CAT_FEATURES,
    FEATURE_COLS,
    MODEL_PATH,
    OUTPUT_DIR,
    TARGET_COL,
    evaluate,
    load_model_input,
    predict,
    train,
)

DATA_PATH = Path('../data/processed/ktx_model_input.csv')
MODEL_OUTPUT_PATH = Path('../models/catboost_model.cbm')
PLOT_OUTPUT_DIR = Path('../data/processed/eda_results')

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 데이터 로드 및 전처리

In [2]:
df = load_model_input(DATA_PATH)

print('shape:', df.shape)
print('columns:')
print(df.columns.tolist())
print('\nfeature_cols:')
print(FEATURE_COLS)
print('target_col:', TARGET_COL)
print('cat_features:', CAT_FEATURES)

display(df.head())

shape: (883, 19)
columns:
['yearmonth', '연도', '월', '분기', '계절', '코로나기간', 'SRT개통후', '노선', '여객수_천명', 'KTX이용률', '공실률', '초과수요', '공실상태', 'KTX전체이용률', 'KTX전체공실률', '개통전', '공실률_lag1', '공실률_lag12', '공실률_ma3']

feature_cols:
['월', '분기', '계절', '코로나기간', 'SRT개통후', '노선', '공실률_lag1', '공실률_lag12', '공실률_ma3']
target_col: 공실률
cat_features: ['노선', '계절']


,yearmonth,연도,월,분기,계절,코로나기간,SRT개통후,노선,여객수_천명,KTX이용률,공실률,초과수요,공실상태,KTX전체이용률,KTX전체공실률,개통전,공실률_lag1,공실률_lag12,공실률_ma3
0,2005-04-01,2005,4,2,봄,0,0,경부선,2123.0,78.0,22.0,0,공실있음,68.0,32.0,0,27.0,30.0,24.333333
1,2005-04-01,2005,4,2,봄,0,0,호남선,412.0,41.0,59.0,0,공실있음,68.0,32.0,0,65.0,62.0,60.333333
2,2005-05-01,2005,5,2,봄,0,0,경부선,2247.0,79.0,21.0,0,공실있음,70.0,30.0,0,22.0,25.0,23.333333
3,2005-05-01,2005,5,2,봄,0,0,호남선,450.0,44.0,56.0,0,공실있음,70.0,30.0,0,59.0,61.0,59.333333
4,2005-06-01,2005,6,2,여름,0,0,경부선,2129.0,77.0,23.0,0,공실있음,68.0,32.0,0,21.0,33.0,23.333333


## 2. Walk-forward Cross Validation 및 학습

In [3]:
# TimeSeriesSplit 사용. 일반 KFold는 사용하지 않는다.
tscv = TimeSeriesSplit(n_splits=5)
print(tscv)

model, metrics_df = train(df)

print('\nFold metrics')
display(metrics_df)

print('\nMean metrics')
display(metrics_df[['MAE', 'MAPE', 'RMSE']].mean().to_frame('mean').T)

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)
0:	learn: 13.9782433	test: 22.6890313	best: 22.6890313 (0)	total: 48.5ms	remaining: 24.2s
100:	learn: 2.6429730	test: 7.8836601	best: 7.8836601 (100)	total: 121ms	remaining: 478ms


200:	learn: 1.8804506	test: 6.6003740	best: 6.6003740 (200)	total: 179ms	remaining: 266ms
300:	learn: 1.4080869	test: 6.3305520	best: 6.3305520 (300)	total: 232ms	remaining: 153ms
400:	learn: 1.1235397	test: 6.2095586	best: 6.2066141 (399)	total: 284ms	remaining: 70.2ms
499:	learn: 0.9098701	test: 6.1342804	best: 6.1325970 (498)	total: 334ms	remaining: 0us

bestTest = 6.132597011
bestIteration = 498

Shrink model to first 499 iterations.


fold 1: MAE=6.1326, MAPE=179.03%, RMSE=7.9886
0:	learn: 15.3767834	test: 17.1496803	best: 17.1496803 (0)	total: 2.07ms	remaining: 1.03s
100:	learn: 3.0135093	test: 7.0199396	best: 7.0199396 (100)	total: 87.3ms	remaining: 345ms


200:	learn: 2.3181462	test: 6.6666592	best: 6.6666592 (200)	total: 161ms	remaining: 240ms
300:	learn: 1.8605080	test: 6.5210219	best: 6.5206140 (297)	total: 238ms	remaining: 158ms
400:	learn: 1.5595713	test: 6.4810110	best: 6.4806018 (396)	total: 316ms	remaining: 78.1ms


499:	learn: 1.3431517	test: 6.4425940	best: 6.4425940 (499)	total: 393ms	remaining: 0us

bestTest = 6.442594018
bestIteration = 499

fold 2: MAE=6.4426, MAPE=67.34%, RMSE=9.1728
0:	learn: 15.1685742	test: 17.7881514	best: 17.7881514 (0)	total: 2.67ms	remaining: 1.33s
100:	learn: 3.5985389	test: 6.8439013	best: 6.8250593 (95)	total: 92.1ms	remaining: 364ms


...

200:	learn: 3.4673735	total: 190ms	remaining: 282ms
300:	learn: 3.0834627	total: 277ms	remaining: 183ms
400:	learn: 2.7802452	total: 367ms	remaining: 90.6ms


499:	learn: 2.5297983	total: 457ms	remaining: 0us

Fold metrics


,fold,MAE,MAPE,RMSE
0,1,6.132598,179.030734,7.988559
1,2,6.442595,67.337270,9.172793
2,3,6.601975,87.802104,9.830165
3,4,11.727766,53.817206,15.732287
4,5,4.074574,37.648597,5.023263



Mean metrics


,MAE,MAPE,RMSE
mean,6.995902,85.127182,9.549413


## 3. 전체 데이터 최종 재학습 모델 저장

In [4]:
MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save_model(MODEL_OUTPUT_PATH)
print(f'saved model: {MODEL_OUTPUT_PATH}')

saved model: ../models/catboost_model.cbm


## 4. 예측값 vs 실제값 및 SHAP Summary 저장

In [5]:
pred_df = evaluate(model, df, PLOT_OUTPUT_DIR)

print(f'saved plot: {PLOT_OUTPUT_DIR / "pred_vs_actual.png"}')
print(f'saved plot: {PLOT_OUTPUT_DIR / "shap_summary.png"}')

display(pred_df.head())

saved plot: ../data/processed/eda_results/pred_vs_actual.png
saved plot: ../data/processed/eda_results/shap_summary.png


,yearmonth,노선,공실률,예측공실률
0,2005-04-01,경부선,22.0,19.908603
1,2005-04-01,호남선,59.0,58.824717
2,2005-05-01,경부선,21.0,17.892254
3,2005-05-01,호남선,56.0,53.398331
4,2005-06-01,경부선,23.0,24.564196


## 5. predict() 함수 확인

In [6]:
sample_pred = predict(model, df.head(5))
display(sample_pred.to_frame())

,예측공실률
0,19.908603
1,58.824717
2,17.892254
3,53.398331
4,24.564196


## 6. Fold 4 상세 분석

Fold 4는 전체 fold 중 MAE가 가장 높으므로, `TimeSeriesSplit(n_splits=5)`의 train/test 기간과 test 구간의 데이터 분포를 별도로 확인한다. 예측값은 전체 데이터로 재학습한 최종 모델이 아니라 Fold 4 train 구간만으로 학습한 모델의 test 예측값이다.

In [7]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

FOLD_TO_ANALYZE = 4
X = df[FEATURE_COLS]
y = df[TARGET_COL]
cat_feature_indices = [X.columns.get_loc(col) for col in CAT_FEATURES]

fold_periods = []
fold4_model = None
fold4_df = None

for fold, (train_idx, test_idx) in enumerate(TimeSeriesSplit(n_splits=5).split(X), start=1):
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    fold_periods.append({
        'fold': fold,
        'train_start': train_df['yearmonth'].min(),
        'train_end': train_df['yearmonth'].max(),
        'test_start': test_df['yearmonth'].min(),
        'test_end': test_df['yearmonth'].max(),
        'train_rows': len(train_df),
        'test_rows': len(test_df),
    })

    if fold == FOLD_TO_ANALYZE:
        fold4_model = CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            eval_metric='MAE',
            early_stopping_rounds=50,
            random_seed=42,
            verbose=False,
        )
        fold4_model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=cat_feature_indices,
            eval_set=(X.iloc[test_idx], y.iloc[test_idx]),
            use_best_model=True,
        )
        fold4_df = test_df.copy()
        fold4_df['예측공실률'] = fold4_model.predict(X.iloc[test_idx])
        fold4_df['오차'] = fold4_df[TARGET_COL] - fold4_df['예측공실률']
        fold4_df['절대오차'] = fold4_df['오차'].abs()

fold_periods_df = pd.DataFrame(fold_periods)
display(fold_periods_df)

fold4_period = fold_periods_df.query('fold == @FOLD_TO_ANALYZE').iloc[0]
print(
    f"Fold 4 train: {fold4_period['train_start'].date()} ~ {fold4_period['train_end'].date()}"
)
print(
    f"Fold 4 test : {fold4_period['test_start'].date()} ~ {fold4_period['test_end'].date()}"
)
print(f"Fold 4 test rows: {fold4_period['test_rows']}")

,fold,train_start,train_end,test_start,test_end,train_rows,test_rows
0,1,2005-04-01,2011-05-01,2011-06-01,2014-12-01,148,147
1,2,2005-04-01,2014-12-01,2014-12-01,2017-08-01,295,147
2,3,2005-04-01,2017-08-01,2017-08-01,2020-02-01,442,147
3,4,2005-04-01,2020-02-01,2020-02-01,2022-07-01,589,147
4,5,2005-04-01,2022-07-01,2022-07-01,2024-12-01,736,147


Fold 4 train: 2005-04-01 ~ 2020-02-01
Fold 4 test : 2020-02-01 ~ 2022-07-01
Fold 4 test rows: 147


In [8]:
fold4_mae = mean_absolute_error(fold4_df[TARGET_COL], fold4_df['예측공실률'])
fold4_rmse = mean_squared_error(fold4_df[TARGET_COL], fold4_df['예측공실률']) ** 0.5

print(f'Fold 4 MAE : {fold4_mae:.4f}')
print(f'Fold 4 RMSE: {fold4_rmse:.4f}')

print('\nFold 4 test 코로나기간 분포')
display(
    fold4_df['코로나기간']
    .value_counts()
    .rename_axis('코로나기간')
    .reset_index(name='건수')
    .sort_values('코로나기간')
)

print('\nFold 4 test 노선별 공실률/예측/오차')
route_summary = (
    fold4_df.groupby('노선')
    .agg(
        실제공실률평균=(TARGET_COL, 'mean'),
        예측공실률평균=('예측공실률', 'mean'),
        MAE=('절대오차', 'mean'),
        건수=(TARGET_COL, 'size'),
    )
    .round(2)
    .sort_values('MAE', ascending=False)
)
display(route_summary)

print('\nFold 4 test 공실상태 분포')
display(
    fold4_df['공실상태']
    .value_counts()
    .rename_axis('공실상태')
    .reset_index(name='건수')
)

print('\n절대오차 상위 10개')
display(
    fold4_df[[
        'yearmonth',
        '노선',
        TARGET_COL,
        '예측공실률',
        '절대오차',
        '코로나기간',
        '공실상태',
    ]]
    .sort_values('절대오차', ascending=False)
    .head(10)
    .round({'예측공실률': 2, '절대오차': 2})
)

Fold 4 MAE : 11.7278
Fold 4 RMSE: 15.7323

Fold 4 test 코로나기간 분포


,코로나기간,건수
1,0,3
0,1,144



Fold 4 test 노선별 공실률/예측/오차


,실제공실률평균,예측공실률평균,MAE,건수
노선,,,,
경부선,37.34,27.32,13.78,29
동해선,24.73,17.42,12.36,30
경전선,26.38,22.00,11.46,29
호남선,42.70,35.24,10.88,30
전라선,27.28,24.55,10.17,29



Fold 4 test 공실상태 분포


,공실상태,건수
0,공실있음,132
1,여유없음,8
2,초과수요,7



절대오차 상위 10개


,yearmonth,노선,공실률,예측공실률,절대오차,코로나기간,공실상태
594,2020-03-01,동해선,71.0,14.10,56.90,1,공실있음
595,2020-03-01,경부선,74.0,26.80,47.20,1,공실있음
596,2020-03-01,경전선,69.0,25.05,43.95,1,공실있음
591,2020-02-01,동해선,36.0,-1.83,37.83,1,공실있음
597,2020-03-01,전라선,61.0,23.64,37.36,1,공실있음
602,2020-04-01,경부선,60.0,26.40,33.60,1,공실있음
643,2021-01-01,경부선,57.0,24.76,32.24,1,공실있음
593,2020-03-01,호남선,68.0,35.84,32.16,1,공실있음
724,2022-05-01,경전선,-12.0,18.66,30.66,1,초과수요
600,2020-04-01,동해선,53.0,23.64,29.36,1,공실있음


In [9]:
from matplotlib import font_manager, rcParams

korean_font_path = Path('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
if korean_font_path.exists():
    font_manager.fontManager.addfont(str(korean_font_path))
    rcParams['font.family'] = font_manager.FontProperties(fname=str(korean_font_path)).get_name()
rcParams['axes.unicode_minus'] = False

fold4_plot_path = PLOT_OUTPUT_DIR / 'fold4_pred_vs_actual.png'
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 8))
for route, route_df in fold4_df.groupby('노선'):
    plt.scatter(
        route_df[TARGET_COL],
        route_df['예측공실률'],
        alpha=0.75,
        label=route,
    )

min_value = min(fold4_df[TARGET_COL].min(), fold4_df['예측공실률'].min())
max_value = max(fold4_df[TARGET_COL].max(), fold4_df['예측공실률'].max())
plt.plot([min_value, max_value], [min_value, max_value], 'r--', label='y=x')
plt.xlabel('Actual vacancy rate')
plt.ylabel('Predicted vacancy rate')
plt.title('Fold 4 Predicted vs Actual Vacancy Rate')
plt.legend(title='Route')
plt.tight_layout()
plt.savefig(fold4_plot_path, dpi=200)
plt.show()

print(f'saved plot: {fold4_plot_path}')

saved plot: ../data/processed/eda_results/fold4_pred_vs_actual.png


### Fold 4 MAE 상승 원인 해석

Fold 4 test 구간은 `2020-02-01`부터 `2022-07-01`까지로, 147행 중 144행이 `코로나기간=1`에 해당한다. 따라서 다른 fold보다 코로나19 충격이 test 데이터에 집중되어 있다. 특히 `2020-03`에는 경부선 74.0%, 동해선 71.0%, 경전선 69.0%, 전라선 61.0%, 호남선 68.0%처럼 모든 노선의 실제 공실률이 급등했지만, Fold 4 모델은 대체로 과거의 정상 운행 패턴을 바탕으로 더 낮게 예측했다.

노선별 평균에서도 Fold 4 test 실제 공실률은 호남선 42.70%, 경부선 37.34%, 전라선 27.28%, 경전선 26.38%, 동해선 24.73%로 높다. 반면 예측 평균은 모든 노선에서 실제보다 낮아, 코로나 초기 수요 급감으로 생긴 공실률 상승을 충분히 따라가지 못했다. 절대오차 상위 사례도 `2020-03`과 `2020-04`에 집중된다.

정리하면 Fold 4 MAE 11.7은 모델 구조 자체의 일반적인 실패라기보다, test 구간 대부분이 코로나 비정상 구간이고 초기 충격의 크기가 lag 변수와 `코로나기간` 단일 플래그만으로 설명하기 어려웠기 때문에 높아진 것으로 해석된다. 단, `TimeSeriesSplit`은 행 단위로 분할되므로 같은 `yearmonth`가 train/test 경계에 걸칠 수 있다. Fold 4에서도 train 종료일과 test 시작일이 모두 `2020-02-01`로 표시된다.

## 7. Part 5 - SHAP 변수 중요도 및 정책 시사점

최종 CatBoost 모델의 SHAP 값을 기준으로 공실률 예측에 크게 기여한 변수를 확인한다. 사이클릭 인코딩 실험은 평균 MAE가 악화되어 제거했고, 아래 결과는 원래 feature 구성으로 복원한 최종 모델 기준이다.

In [10]:
import shap

FEATURE_EN_NAMES = {
    '공실률_lag1': 'vacancy_lag1',
    '공실률_ma3': 'vacancy_ma3',
    '월': 'month',
    '코로나기간': 'covid_period',
    '분기': 'quarter',
    '공실률_lag12': 'vacancy_lag12',
    '노선': 'route',
    'SRT개통후': 'after_srt',
    '계절': 'season',
}
FEATURE_KO_NAMES = {
    'vacancy_lag1': '전월 공실률',
    'vacancy_ma3': '3개월 이동평균',
    'month': '월',
    'covid_period': '코로나기간',
    'quarter': '분기',
    'vacancy_lag12': '전년 동월 공실률',
    'route': '노선',
    'after_srt': 'SRT 개통 후',
    'season': '계절',
}
feature_positions = {feature: idx for idx, feature in enumerate(FEATURE_COLS)}

X = df[FEATURE_COLS]
explainer = shap.TreeExplainer(model)
shap_values_raw = explainer.shap_values(X)
shap_values = shap_values_raw[1] if isinstance(shap_values_raw, list) else shap_values_raw

shap_importance = (
    pd.DataFrame({
        'feature': FEATURE_COLS,
        'mean_abs_shap': abs(shap_values).mean(axis=0),
    })
    .assign(
        feature_en=lambda d: d['feature'].map(FEATURE_EN_NAMES),
        변수명=lambda d: d['feature_en'].map(FEATURE_KO_NAMES),
    )
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)
shap_importance['순위'] = shap_importance.index + 1

shap_top5 = shap_importance.loc[
    :4,
    ['순위', '변수명', 'feature_en', 'feature', 'mean_abs_shap'],
].rename(columns={
    'feature_en': '영문 변수명',
    'feature': '원본 컬럼명',
    'mean_abs_shap': '평균 |SHAP|',
})

print('SHAP 변수 중요도 상위 5개')
display(shap_top5.round({'평균 |SHAP|': 3}))

SHAP 변수 중요도 상위 5개


,순위,변수명,영문 변수명,원본 컬럼명,평균 |SHAP|
0,1,전월 공실률,vacancy_lag1,공실률_lag1,9.903
1,2,3개월 이동평균,vacancy_ma3,공실률_ma3,4.810
2,3,월,month,월,1.756
3,4,코로나기간,covid_period,코로나기간,1.715
4,5,분기,quarter,분기,1.528


In [11]:
direction_rows = []
for row in shap_importance.head(5).itertuples(index=False):
    feature = row.feature
    feature_name = row.변수명
    shap_series = pd.Series(
        shap_values[:, feature_positions[feature]],
        index=X.index,
        name='shap',
    )
    value_series = X[feature]

    if feature == '코로나기간':
        grouped = (
            pd.DataFrame({'value': value_series, 'shap': shap_series})
            .groupby('value')['shap']
            .mean()
            .round(3)
            .to_dict()
        )
        direction = f"코로나기간=1의 평균 SHAP {grouped.get(1, float('nan')):.3f}, 비코로나기간=0의 평균 SHAP {grouped.get(0, float('nan')):.3f}"
    elif feature in ['월', '분기']:
        grouped = (
            pd.DataFrame({'value': value_series, 'shap': shap_series})
            .groupby('value')['shap']
            .mean()
            .round(3)
        )
        direction = f"가장 높이는 값 {grouped.idxmax()} (SHAP {grouped.max():.3f}), 가장 낮추는 값 {grouped.idxmin()} (SHAP {grouped.min():.3f})"
    elif pd.api.types.is_numeric_dtype(value_series):
        corr = value_series.corr(shap_series)
        direction = f"값과 SHAP의 상관계수 {corr:.3f}"
    else:
        grouped = (
            pd.DataFrame({'value': value_series, 'shap': shap_series})
            .groupby('value')['shap']
            .mean()
            .sort_values(ascending=False)
            .round(3)
            .to_dict()
        )
        direction = str(grouped)

    direction_rows.append({
        '변수명': feature_name,
        '방향성 근거': direction,
    })

direction_df = pd.DataFrame(direction_rows)
display(direction_df)

,변수명,방향성 근거
0,전월 공실률,값과 SHAP의 상관계수 0.989
1,3개월 이동평균,값과 SHAP의 상관계수 0.917
2,월,"가장 높이는 값 6 (SHAP 2.823), 가장 낮추는 값 5 (SHAP -2.553)"
3,코로나기간,"코로나기간=1의 평균 SHAP 5.330, 비코로나기간=0의 평균 SHAP -0.964"
4,분기,"가장 높이는 값 1 (SHAP 2.927), 가장 낮추는 값 4 (SHAP -2.036)"


### 변수별 영향 방향 해석

- **전월 공실률**: SHAP 중요도 1위이며, 값과 SHAP의 상관계수가 약 `0.989`로 매우 높다. 전월 공실률이 높을수록 이번 달 공실률도 높아지는 방향으로 작동한다.
- **3개월 이동평균**: SHAP 중요도 2위이며, 값과 SHAP의 상관계수가 약 `0.917`이다. 최근 3개월 공실률 평균이 높을수록 이번 달 공실률 예측값도 높아진다.
- **월**: 특정 월 효과가 존재한다. 월별 평균 SHAP 기준으로 `6월`, `12월`, `7월`, `3월`, `1월`은 공실률을 높이는 방향이고, `5월`, `4월`, `2월`, `10월`, `11월`은 낮추는 방향으로 나타난다.
- **코로나기간**: `코로나기간=1`의 평균 SHAP은 약 `+5.330`, `코로나기간=0`의 평균 SHAP은 약 `-0.964`이다. 코로나기간에는 공실률이 평균 대비 크게 증가하는 방향으로 작동한다.
- **분기**: 1분기는 공실률을 높이는 방향, 4분기는 낮추는 방향으로 나타난다. 다만 `월` 변수가 함께 들어가 있으므로 분기 효과는 월별 패턴을 보조적으로 설명하는 변수로 해석하는 것이 적절하다.

### 사이클릭 인코딩 실험과 비교

사이클릭 인코딩 전 평균 MAE는 `6.9959`였고, 사이클릭 인코딩 후 평균 MAE는 `7.0493`이었다. 따라서 사이클릭 인코딩은 전체 평균 MAE 기준으로 개선되지 않아 제거했다. 원래 feature 구성으로 되돌린 뒤 평균 MAE `6.9959`와 SHAP 상위 순위가 복원됐다.

### Part 2 EDA와 연결한 해석

Part 2 EDA에서 연도 트렌드와 코로나 구간의 구조적 변화가 계절 차이보다 더 중요한 패턴으로 확인됐다. SHAP 결과에서도 `전월 공실률`과 `3개월 이동평균`이 1, 2위를 차지해, 계절 자체보다 직전 상태와 최근 추세가 가장 강력한 예측 변수임을 확인했다. 또한 `코로나기간`이 상위 5개 변수에 포함되어, EDA에서 관찰한 코로나 시기 공실률 급등이 모델 해석에서도 중요한 요인으로 재확인된다.

### 정책 시사점

1. **최근 공실률 추세 기반의 조기 경보 지표 운영**  
   전월 공실률과 3개월 이동평균이 가장 중요한 변수이므로, 노선별 최근 1~3개월 공실률이 연속 상승하는 구간을 조기 경보 대상으로 지정한다. 해당 구간은 할인, 관광 연계 상품, 운행 편성 조정 검토 대상으로 우선 분류할 수 있다.

2. **월별 수요 패턴을 반영한 사전 프로모션 설계**  
   월 변수가 상위 변수로 나타나므로, 월별로 반복되는 공실률 상승 구간에는 사후 대응보다 사전 프로모션이 적합하다. 예를 들어 공실률 상승 방향의 월에는 노선별 할인율, 묶음 상품, 시간대별 좌석 배분 정책을 미리 준비한다.

3. **비정상 충격 구간은 별도 운영 규칙 적용**  
   코로나기간 변수의 영향이 크다는 점은 감염병, 대규모 재난, 운행 제한 같은 비정상 이벤트가 일반적인 계절 패턴보다 큰 영향을 줄 수 있음을 의미한다. 이런 구간은 평시 모델 예측만 적용하지 말고, 이벤트 플래그와 수요 급변 감지 기준을 별도로 두어 임시 감편, 환불 정책, 대체 수요 유도 전략을 함께 검토해야 한다.